In [ ]:
import pandas as pd
import numpy as np
import faiss
import pickle
from sentence_transformers import SentenceTransformer


C:\Users\kaush\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_prod= pd.read_csv("data/products.csv")
df_faq=pd.read_csv("data/fact-base-tesco.csv")


In [3]:
df_prod

,id,title,description,bulkBuyLimit,status,price_actual,price_unitPrice,price_unitOfMeasure,promotion_id,promotion_startDate,promotion_endDate,promotion_description
0,274510599,Tesco 0% Fat Greek Style Yogurt 500G,Fat free Greek style natural yogurt.,16,AvailableForSale,1.20,0.24,100g,NaN,NaN,NaN,NaN
1,292225110,Suntrail Farms Price Orange Minimum 5 Pack,Oranges.,16,ExcludedProduct,0.99,0.20,each,NaN,NaN,NaN,NaN
2,256651384,Tesco Asparagus Tips 125G,Asparagus,16,AvailableForSale,2.00,16.00,kg,NaN,NaN,NaN,NaN
3,312704710,Tesco Iceberg Lettuce 200G,Cut iceberg lettuce leaves.,25,AvailableForSale,0.70,0.35,100g,NaN,NaN,NaN,NaN
4,312352060,Tesco Butterhead Salad 80G,Green and red butterhead lettuce with baby spi...,25,AvailableForSale,1.00,1.25,100g,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
7228,316422645,Loacker Chocolaterie Dark Chocolate Wafer Bars...,Crispy wafers with dark chocolate cream fillin...,25,AvailableForSale,2.00,2.86,100g,NaN,NaN,NaN,NaN
7229,318175784,Celebrations Milk Chocolate Easter Egg 189g,Milk chocolate.\nAn assortment of milk chocola...,99,AvailableForSale,4.00,2.12,100g,NaN,NaN,NaN,NaN
7230,315579328,Smarties Sea Splash Chocolate Easter Egg 226G,Milk chocolate egg with 2x tubes of Smarties® ...,99,AvailableForSale,6.00,2.66,100g,NaN,NaN,NaN,NaN
7231,314842427,Fruit Bowl Yogurt Flakes Raspberry 5x18g,Fruit pieces made with concentrated apple and ...,25,AvailableForSale,2.25,25.00,kg,90827651.0,2024-12-27T00:00:00Z,2025-01-22T00:00:00Z,£1.70 Clubcard Price


In [4]:
df_faq

,ID,Topic,Subtopic,Question,Answer
0,1.0,Online Grocery FAQs,Delivery and collection basics,Where Tesco delivers to,We deliver to most UK residential addresses. T...
1,2.0,Online Grocery FAQs,Delivery and collection basics,Delivery and Click+Collect prices,"The standard delivery charge is between £3–£7,..."
2,3.0,Online Grocery FAQs,Delivery and collection basics,Minimum order value,A £5 minimum basket charge will be added to de...
3,4.0,Online Grocery FAQs,Delivery and collection basics,Returning an item,Please see our returns policy.
4,5.0,Online Grocery FAQs,Ordering online,Slot times and options,You can choose to get your shopping delivered ...
...,...,...,...,...,...
92,NaN,Online Grocery FAQs,Ratings and reviews,Deleting my review,"If you’ve shared reviews on our websites, you’..."
93,NaN,Online Grocery FAQs,Ratings and reviews,Reporting a review,The report button below a review allows you to...
94,NaN,Online Grocery FAQs,Ratings and reviews,Syndicated reviews,These reviews are existing customer reviews su...
95,NaN,Online Grocery FAQs,Accessibility,Accessibility online\n,We try to make all of our websites and apps ac...


### Loading Embedding model (all-MiniLM-L6-v2)

In [6]:
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded")

C:\Users\kaush\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kaush\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7633.15it/s]
BertModel LOAD RE

Model loaded


## FAQ Index

In [7]:
# Build chunks — Topic + Subtopic gives the retriever context
def build_faq_chunk(row):
    return (
        f"Topic: {row['Topic']} | "
        f"Subtopic: {row['Subtopic']} | "
        f"Question: {row['Question']} | "
        f"Answer: {row['Answer']}"
    )

faq_chunks = df_faq.apply(build_faq_chunk, axis=1).tolist()

print(f"FAQ chunks: {len(faq_chunks)}")
print(f"Sample: {faq_chunks[0][:200]}")

FAQ chunks: 97
Sample: Topic: Online Grocery FAQs | Subtopic: Delivery and collection basics | Question: Where Tesco delivers to | Answer: We deliver to most UK residential addresses. To check whether we deliver to your add


In [8]:
# Embed FAQ chunks
print("Embedding FAQ chunks...")
faq_vectors = embed_model.encode(faq_chunks, show_progress_bar=True)
print(f"FAQ vectors shape: {faq_vectors.shape}")

Embedding FAQ chunks...


Batches: 100%|██████████| 4/4 [00:02<00:00,  1.54it/s]

FAQ vectors shape: (97, 384)


In [9]:
# Build FAISS index
faq_index = faiss.IndexFlatL2(384)
faq_index.add(np.array(faq_vectors, dtype='float32'))
print(f"FAQ index size: {faq_index.ntotal}")

FAQ index size: 97


In [10]:
# Save index and chunks
faiss.write_index(faq_index, 'indexes/faiss_faq_index')
with open('indexes/faq_chunks.pkl', 'wb') as f:
    pickle.dump(faq_chunks, f)

print("FAQ index saved")

FAQ index saved


## Product Index

In [11]:
# Build chunks — handle promotion nulls gracefully
def build_product_chunk(row):
    promo = (
        f" | Promotion: {row['promotion_description']}"
        if pd.notna(row['promotion_description'])
        else ""
    )
    return (
        f"Product: {row['title']} | "
        f"Description: {row['description']} | "
        f"Price: £{row['price_actual']} | "
        f"Unit price: £{row['price_unitPrice']} per {row['price_unitOfMeasure']} | "
        f"Status: {row['status']}"
        f"{promo}"
    )

product_chunks = df_prod.apply(build_product_chunk, axis=1).tolist()

print(f"Product chunks: {len(product_chunks)}")
print(f"Sample: {product_chunks[0][:200]}")

Product chunks: 7233
Sample: Product: Tesco 0% Fat Greek Style Yogurt 500G | Description: Fat free Greek style natural yogurt. | Price: £1.2 | Unit price: £0.24 per 100g | Status: AvailableForSale


In [12]:
# Embed product chunks — 7233 rows, will take 1-2 minutes
print("Embedding product chunks...")
product_vectors = embed_model.encode(
    product_chunks, 
    show_progress_bar=True,
    batch_size=64  # process in batches to avoid memory issues
)

Embedding product chunks...


Batches: 100%|██████████| 114/114 [01:21<00:00,  1.39it/s]


In [13]:
print(f"Product vectors shape: {product_vectors.shape}")

Product vectors shape: (7233, 384)


In [14]:
# Build FAISS index
product_index = faiss.IndexFlatL2(384)
product_index.add(np.array(product_vectors, dtype='float32'))
print(f"Product index size: {product_index.ntotal}")

# Save index and chunks
faiss.write_index(product_index, 'indexes/faiss_product_index')
with open('indexes/product_chunks.pkl', 'wb') as f:
    pickle.dump(product_chunks, f)

print("Product index saved")

Product index size: 7233
Product index saved


## Retrieval Test

In [15]:
def test_retrieval(query, index, chunks, top_k=3):
    query_vector = embed_model.encode([query])
    distances, indices = index.search(
        np.array(query_vector, dtype='float32'), top_k
    )
    print(f"\nQuery: '{query}'")
    for i, idx in enumerate(indices[0]):
        print(f"\nResult {i+1}: {chunks[idx][:150]}")

# Test FAQ retrieval
test_retrieval("What is the refund policy?", faq_index, faq_chunks)

# Test product retrieval
test_retrieval("low fat yogurt under £2", product_index, product_chunks)


Query: 'What is the refund policy?'

Result 1: Topic: Online Grocery FAQs | Subtopic: Collecting your order and delivery issues | Question: I've received a damaged item | Answer: If an item is dama

Result 2: Topic: Online Grocery FAQs | Subtopic: Delivery and collection basics | Question: Returning an item | Answer: Please see our returns policy.

Result 3: Topic: Online Grocery FAQs | Subtopic: Whoosh | Question: Returning items | Answer: You can return unwanted items to your local store, and we’ll issue

Query: 'low fat yogurt under £2'

Result 1: Product: Creamfields Low Fat Natural Yogurt 500G | Description: Low fat natural yogurt. | Price: £0.55 | Unit price: £0.11 per 100g | Status: Availabl

Result 2: Product: Muller Light Strawberry Yogurt 160G | Description: Strawberry fat free yogurt with sweetener and added vitamins B6 & D | Enjoy as part of a v

Result 3: Product: Creamfields Strawberry Low Fat Yogurt 450G | Description: Stabilised, sweetened low fat yogurt blended with s